# 総研大 2024年度 講義「シミュレーション天文学」 / SOKEN-dai FY2024 Lecture Course: “Simulation Astronomy”
### 担当：馬場淳一 (国立天文台 科学研究部/鹿児島大学 天の川銀河研究センター) / Junichi Baba (D. of Science, NAOJ / Kagoshima University)

This code is provided for educational purposes only. Users are encouraged to utilize it at their own risk. I take no responsibility for any results obtained from research or analyses conducted using this code. It is essential to deepen your understanding and verify the results independently. Additionally, it is recommended to cross-reference with other reliable resources to ensure the appropriate methodologies are applied.

このコードは教育目的で提供されています。使用にあたっては、自己責任で行ってください。提供されたコードを基に行った研究や解析によって得られた結果について、私は一切の責任を負いません。使用する際には、必ず自身の理解を深め、結果を検証してください。また、他の信頼できるリソースと照らし合わせて、適切な方法論を選択することをお勧めします。

## 学習目標と計算上の前提

Heun法と陽的中点法は大域2次精度です。線形減衰では同一の安定性多項式ですが、非線形問題では結果が異なります。解析解の比較例は t0=0, y0=1 です。

検証課題と判定基準は [演習ガイド](../docs/exercises.md) を参照してください。


In [ ]:
import numpy as np
import time 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
parameters = {
    'axes.labelsize': 26,           # Set the font size of the x-axis and y-axis labels to 26
    'xtick.labelsize': 24,          # Set the font size of the x-axis tick labels to 24
    'ytick.labelsize': 24,          # Set the font size of the y-axis tick labels to 24
    'axes.titlesize': 24,           # Set the font size of the axis titles (not the overall graph title) to 24
    'figure.titlesize': 24,         # Set the font size of the overall graph title to 24
    'legend.fontsize': 22,          # Set the font size of the legend to 22
    'figure.figsize': [10, 6],      # Set the default figure size (width 10 inches, height 6 inches)
    'axes.linewidth': 2.0           # Set the linewidth of the axes
}
plt.rcParams.update(parameters)

In [ ]:
def f(t, y, alpha):
    """
    Defines the right-hand side of the differential equation dy/dt = -alpha * y.

    Parameters:
        t (float): The current time
        y (float): The current value of y
        alpha (float): The coefficient alpha in the equation

    Returns:
        float: The value of dy/dt
    """
    return -alpha * y

def analytical_solution(t, alpha):
    """
    Computes the analytical solution y(t) = e^(-alpha * t).

    Parameters:
        t (float or np.ndarray): The time at which to evaluate the solution
        alpha (float): The coefficient alpha in the equation

    Returns:
        float or np.ndarray: The analytical solution y(t)
    """
    return np.exp(-alpha * t)

## Explanation of the `euler_method` Function

This function implements the **explicit Forward Euler method** for solving ordinary differential equations (ODEs) of the form \( \frac{dy}{dt} = f(t, y) \). The Forward Euler method is an **explicit** method, meaning it uses the current information at each time step to compute the next value directly.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and the step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **Main Loop for Time Stepping**:
   - For each time step \( n \), the current time \( t_n \) and the corresponding solution \( y_n \) are retrieved.
   - The next value \( y_{\text{new}} \) is computed using the Forward Euler formula:
     \[
     y_{\text{new}} = y_n + h \cdot f(t_n, y_n, \alpha)
     \]
     This formula updates the solution at the next time step by using the current derivative information from \( f(t_n, y_n, \alpha) \).

3. **Update the Solution**:
   - Once \( y_{\text{new}} \) is computed, it is stored in `y_values[n+1]`.

4. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **Time stepping**: The solution progresses step-by-step from \( t_0 \) to \( t_{\text{end}} \), updating the solution using the Forward Euler method formula.
2. **Explicit Update**: At each time step, the current solution \( y_n \) and the function value \( f(t_n, y_n, \alpha) \) are used to compute the next value \( y_{\text{new}} \).
3. **Simple and Direct**: The Forward Euler method is straightforward but can become unstable for stiff ODEs or when the step size \( h \) is too large.

### Key Advantages:
- **Simple and Easy to Implement**: The Forward Euler method is an explicit method, making it computationally cheap and easy to implement.
- **Direct Calculation**: The next solution value is calculated directly from the current information without needing to solve an implicit equation.
  
### Limitations:
- **Limited Stability**: The Forward Euler method may become unstable for large step sizes or for stiff ODEs, meaning that smaller step sizes are often required for accurate results.
- **First-Order Accuracy**: The method has first-order accuracy, meaning the error decreases linearly as the step size is reduced.

In [ ]:
# Define the Forward Euler Method Function
def euler_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the explicit Forward Euler method.

    Parameters:
        f (callable): Function representing dy/dt = f(t, y)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]
        y_new = y_n + h * f(t_n, y_n, alpha)
        y_values[n + 1] = y_new

    return t_values, y_values

# Heun's Method (2nd-order Runge-Kutta Method) Explanation

### Objective
**Heun's Method**, also known as the **Improved Euler Method** or **Modified Euler Method**, is a second-order numerical technique for solving **ordinary differential equations (ODEs)**. It improves upon the simple **Forward Euler Method** by incorporating a predictor-corrector mechanism, which provides more accuracy while still being easy to implement.

### Key Equation

The general form of an initial value problem is:

$$
\frac{dy}{dt} = f(t, y), \quad y(t_0) = y_0
$$

Here, $y(t)$ is the function we want to approximate, $t$ is the independent variable (often representing time), and $f(t, y)$ represents the derivative of $y$ with respect to $t$.

Heun's method combines a **predictor step** (using the Forward Euler Method) and a **corrector step** to improve the accuracy. The method is expressed as follows:

1. **Predictor (Forward Euler Step)**:
   $$
   y_{\text{pred}} = y_n + h \cdot f(t_n, y_n)
   $$

2. **Corrector (Average of slopes)**:
   $$
   y_{n+1} = y_n + \frac{h}{2} \left[ f(t_n, y_n) + f(t_{n+1}, y_{\text{pred}}) \right]
   $$

Where:
- $y_{\text{pred}}$ is the predicted value from the Euler step.
- $y_{n+1}$ is the corrected value using the average slope.

### Step-by-Step Procedure

1. **Initialization**:
   - Start with an initial condition: $y(t_0) = y_0$.
   - Set the step size $h$ and the time range for the solution.

2. **Predictor Step**:
   - Compute the predicted value $y_{\text{pred}}$ using:
     $$
     y_{\text{pred}} = y_n + h \cdot f(t_n, y_n)
     $$

3. **Corrector Step**:
   - Use the average of the slopes to compute the corrected value $y_{n+1}$:
     $$
     y_{n+1} = y_n + \frac{h}{2} \left[ f(t_n, y_n) + f(t_{n+1}, y_{\text{pred}}) \right]
     $$

4. **Iteration**:
   - Update $t_n$ to $t_{n+1} = t_n + h$, and repeat the predictor and corrector steps for each time step.


### Advantages
- **Increased Accuracy**: Heun’s method is more accurate than the Forward Euler Method because it uses both the initial slope and a corrected slope.
- **Easy to Implement**: Despite the improved accuracy, Heun's method remains relatively simple to implement.

### Disadvantages
- **Moderate Computational Cost**: Heun's method requires two evaluations of the function $f(t, y)$ at each time step, which makes it computationally more expensive than the Forward Euler Method.

### Summary
Heun's Method is a second-order method that improves upon the Forward Euler Method by averaging the initial and predicted slopes. It strikes a good balance between accuracy and simplicity, making it a useful method for solving ODEs when moderate precision is required.

## Explanation of the `rk2_heun_method` Function

This function implements **Heun's method**, which is an improved version of the Euler method, also known as the **second-order Runge-Kutta (RK2)** method. It improves upon the basic Forward Euler method by using an additional step to correct the predicted value, resulting in greater accuracy. The method consists of two steps: a **predictor step** and a **corrector step**.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **Main Loop for Time Stepping**:
   - For each time step \( n \), the current time \( t_n \) and the corresponding solution \( y_n \) are retrieved.

3. **Predictor Step**:
   - A **Forward Euler step** is first performed to calculate an initial guess for the next solution point:
     \[
     y_{\text{predictor}} = y_n + h \cdot f(t_n, y_n, \alpha)
     \]
   - This predicted value \( y_{\text{predictor}} \) serves as an estimate of the solution at \( t_{\text{new}} = t_n + h \).

4. **Corrector Step**:
   - The corrector step improves the solution by taking the **average slope** between the current slope and the slope at the predicted point:
     \[
     y_{\text{corrector}} = y_n + \frac{h}{2} \cdot \left( f(t_n, y_n, \alpha) + f(t_{\text{new}}, y_{\text{predictor}}, \alpha) \right)
     \]
   - This correction adjusts the predicted value using the average of the initial and predicted slopes.

5. **Update the Solution**:
   - The corrected value \( y_{\text{corrector}} \) is stored in `y_values[n+1]`.

6. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **Predictor Step**: First, the method uses the Forward Euler step to predict the next value using the slope at the current point.
2. **Corrector Step**: Then, it corrects the predicted value by averaging the slopes from both the current point and the predicted point, increasing the accuracy.
3. **Improved Accuracy**: By including the corrector step, Heun’s method achieves second-order accuracy, offering a significant improvement over the basic Forward Euler method.

### Key Advantages:
- **Improved Accuracy**: Heun’s method is more accurate than the standard Euler method because it accounts for changes in the slope over the time step.
- **Second-Order Accuracy**: The method has second-order accuracy, meaning the error decreases quadratically as the step size \( h \) is reduced.
  
### Limitations:
- **More Computational Effort**: While more accurate, Heun’s method requires two evaluations of the function \( f \) per time step (for both the predictor and corrector), making it more computationally expensive than the Forward Euler method.

In [ ]:
def rk2_heun_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using Heun's method (improved Euler method).

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]

        # Predictor step (Forward Euler)
        y_predictor = y_n + h * f(t_n, y_n, alpha)

        # Corrector step (average slope)
        t_new = t_n + h
        y_corrector = y_n + (h / 2) * (f(t_n, y_n, alpha) + f(t_new, y_predictor, alpha))

        y_values[n + 1] = y_corrector

    return t_values, y_values

# Modifiled Euler Method (2nd-order Runge-Kutta Method) Explanation

### Objective
The **Modifiled Euler Method**, also known as the **Euler-Cauchy Method**, is an improvement over the **Forward Euler Method** for solving **ordinary differential equations (ODEs)**. Like **Heun's Method**, it utilizes a predictor-corrector approach, providing more accurate solutions compared to the simple Forward Euler Method. It is a second-order method, meaning it is more accurate for the same step size than the first-order Forward Euler Method.

### Key Equation

For a general initial value problem of the form:

$$
\frac{dy}{dt} = f(t, y), \quad y(t_0) = y_0
$$

where $y(t)$ is the unknown function, $t$ is the independent variable (time), and $f(t, y)$ is the derivative of $y$ with respect to $t$.

The Modified Euler Method involves two main steps:

1. **Predictor Step (Euler's Forward Method)**:
   $$
   y_{\text{pred}} = y_n + h \cdot f(t_n, y_n)
   $$

2. **Corrector Step**:
   $$
   y_{n+1} = y_n + \frac{h}{2} \left[ f(t_n, y_n) + f(t_{n+1}, y_{\text{pred}}) \right]
   $$

Where:
- $y_{\text{pred}}$ is the predicted value of $y$ at the next time step using the Forward Euler Method.
- $y_{n+1}$ is the corrected value of $y$.
- $h$ is the time step size.
- $f(t, y)$ is the function defining the ODE.

### Step-by-Step Procedure

1. **Initialization**:
   - Start with an initial condition: $y(t_0) = y_0$.
   - Set the step size $h$ and determine the time range over which the solution is desired.

2. **Predictor Step**:
   - Compute the predicted value of $y$ using the Forward Euler Method:
     $$
     y_{\text{pred}} = y_n + h \cdot f(t_n, y_n)
     $$

3. **Corrector Step**:
   - Compute the corrected value using the average of the slope at the current point and the predicted point:
     $$
     y_{n+1} = y_n + \frac{h}{2} \left[ f(t_n, y_n) + f(t_{n+1}, y_{\text{pred}}) \right]
     $$

4. **Iteration**:
   - Repeat the predictor and corrector steps for each time step until the final time is reached.

### Advantages
- **More Accurate than Euler’s Method**: The Modified Euler Method provides greater accuracy than the Forward Euler Method due to the use of both the initial slope and the corrected slope.
- **Simplicity**: The method retains a straightforward structure, making it easy to implement while offering improved precision.

### Disadvantages
- **Moderate Computational Cost**: Similar to Heun's Method, the Modified Euler Method requires two evaluations of the function $f(t, y)$ per step, which increases computational effort compared to the Forward Euler Method.

## Explanation of the `rk2_midpoint_method` Function

This function implements the **midpoint method**, which is a second-order **Runge-Kutta (RK2)** method for solving ordinary differential equations (ODEs) of the form \( \frac{dy}{dt} = f(t, y) \). The midpoint method improves upon the basic Euler method by using an intermediate value (midpoint) to compute a more accurate slope, providing better accuracy.

### Function Parameters:
- **`f` (callable)**: The ODE function representing \( f(t, y, \alpha) \), where \( \alpha \) is a parameter of the ODE.
- **`y0` (float)**: The initial condition, representing \( y(t_0) = y_0 \).
- **`t0` (float)**: The starting time.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization.
- **`alpha` (float)**: A parameter of the ODE that influences the behavior of \( f \).

### Returns:
- **`t_values` (numpy.ndarray)**: An array containing the time points from \( t_0 \) to \( t_{\text{end}} \).
- **`y_values` (numpy.ndarray)**: An array containing the numerical solution for \( y \) at each time point.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps \( N \) is calculated based on the total simulation time and the step size.
   - The `t_values` array holds the time points, and `y_values` stores the corresponding \( y \)-values.
   - The initial condition \( y_0 \) is assigned to `y_values[0]`.

2. **Main Loop for Time Stepping**:
   - For each time step \( n \), the current time \( t_n \) and the corresponding solution \( y_n \) are retrieved.

3. **Predictor Step** (Euler's step to midpoint):
   - The first slope \( k_1 \) is computed using the initial slope at \( t_n \) and \( y_n \):
     \[
     k_1 = f(t_n, y_n, \alpha)
     \]
   - This slope is used to predict the midpoint value \( y_{\text{mid}} \):
     \[
     y_{\text{mid}} = y_n + \frac{h}{2} \cdot k_1
     \]
   - The midpoint value gives an estimate of \( y \) at the midpoint of the interval.

4. **Corrector Step** (using slope at the midpoint):
   - The second slope \( k_2 \) is computed using the function evaluated at the midpoint \( t_{\text{mid}} = t_n + \frac{h}{2} \):
     \[
     k_2 = f\left( t_n + \frac{h}{2}, y_{\text{mid}}, \alpha \right)
     \]
   - The corrected value \( y_{\text{new}} \) is then calculated using this slope:
     \[
     y_{\text{new}} = y_n + h \cdot k_2
     \]

5. **Update the Solution**:
   - The corrected value \( y_{\text{new}} \) is stored in `y_values[n+1]`.

6. **Final Output**:
   - The function returns the `t_values` (time points) and `y_values` (corresponding solutions at each time point).

### Example Workflow:
1. **Predictor Step (Midpoint)**: The midpoint method first predicts an intermediate value \( y_{\text{mid}} \) by taking an Euler step using the initial slope \( k_1 \).
2. **Corrector Step**: The slope at the midpoint \( k_2 \) is used to correct the solution, providing a more accurate estimate of the solution at the next time point.
3. **Improved Accuracy**: By using the slope at the midpoint rather than at the start of the step, the midpoint method achieves second-order accuracy.

### Key Advantages:
- **Second-Order Accuracy**: The midpoint method is a second-order Runge-Kutta method, offering better accuracy than the standard Euler method.
- **Use of Midpoint**: By evaluating the slope at the midpoint, this method provides a more accurate estimate of the next solution value.

### Limitations:
- **More Computational Effort**: Similar to other second-order methods, this method requires two function evaluations per time step, making it computationally more expensive than the Forward Euler method.

In [ ]:
def rk2_midpoint_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the RK2 method (midpoint method).

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]

        # Predictor step (Euler's step to midpoint)
        k1 = f(t_n, y_n, alpha)
        y_mid = y_n + (h / 2) * k1  # Midpoint value

        # Corrector step (use slope at the midpoint)
        k2 = f(t_n + h / 2, y_mid, alpha)
        y_new = y_n + h * k2  # Update using the slope at midpoint

        y_values[n + 1] = y_new

    return t_values, y_values

In [ ]:
# Main function to execute the comparison and analysis
def main():
    # Initial conditions and settings
    y0 = 1.0
    t0 = 0.0
    t_end = 1.0
    step_sizes = [0.1, 0.05, 0.025, 0.0125, 0.00625, 0.003125, 0.0015625, 
                  0.00078125, 0.000390625, 0.0001953125, 0.0001]
    alphas = [1]  # Different alpha values for comparison
    
    # Prepare for plotting
    size = 3
    plt.figure(figsize=(4 * size, 3 * size))
    
    # Define colors for different alpha values
    colors = cm.viridis(np.linspace(0, 1, len(alphas)))
    
    # Iterate over different alpha values
    for idx, alpha in enumerate(alphas):
        # Calculate the analytical solution at t_end
        y_true = analytical_solution(t_end, alpha)
        print(f"Analytical solution y({t_end}) with alpha={alpha} = {y_true:.10f}\n")
        
        # Lists to store errors for each method and step size
        euler_errors = []
        heun_errors = []
        midpoint_errors = []
        
        # Print header
        print(f"Comparison of Methods for alpha = {alpha}:")
        print(f"{'Step Size h':>15} | {'Euler Error':>15} | {'Heun Error':>15} | {'Midpoint Error':>15}")
        print("-" * 75)
        
        # Apply each method for every step size
        for h in step_sizes:
            # Euler Method
            _, y_values_euler = euler_method(f, y0, t0, t_end, h, alpha)
            y_euler = y_values_euler[-1]
            euler_error = abs(y_euler - y_true) / abs(y_true)
            euler_errors.append(euler_error)

            # Heun Method
            _, y_values_heun = rk2_heun_method(f, y0, t0, t_end, h, alpha)
            y_heun = y_values_heun[-1]
            heun_error = abs(y_heun - y_true) / abs(y_true)
            heun_errors.append(heun_error)

            # Midpoint Method
            _, y_values_midpoint = rk2_midpoint_method(f, y0, t0, t_end, h, alpha)
            y_midpoint = y_values_midpoint[-1]
            midpoint_error = abs(y_midpoint - y_true) / abs(y_true)
            midpoint_errors.append(midpoint_error)

            # Print results for each step size
            print(f"{h:15.5e} | {euler_error:15.10e} | {heun_error:15.10e} | {midpoint_error:15.10e}")
        
        print("\n")  # Add space between different alpha values
        
        # Plot relative errors for each method on a log-log scale
        plt.loglog(step_sizes, euler_errors, 'o-',  markersize=10, lw=4, color=colors[idx], label=f'Euler alpha={alpha}')
        plt.loglog(step_sizes, heun_errors, 'x--',  markersize=10, lw=4, color=colors[idx], label=f'Heun alpha={alpha}')
        plt.loglog(step_sizes, midpoint_errors, 's-',  markersize=10, lw=4, color=colors[idx], label=f'Midpoint alpha={alpha}')
    
    plt.xlabel('Step Size h')
    plt.ylabel('Relative Error')
    plt.title('Comparison of Euler, Heun, and Midpoint Methods')
    plt.legend()
    plt.grid(True, which="both", ls="--")
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
# Main function to execute the comparison and analysis
def main():
    # Initial conditions and settings
    y0 = 1.0
    t0 = 0.0
    t_end = 1.0
    step_sizes = [0.1, 0.05, 0.025, 0.0125, 0.00625, 0.003125, 0.0015625, 
                  0.00078125, 0.000390625, 0.0001953125, 0.0001]
    alphas = [1]  # Different alpha values for comparison
    
    # Prepare for plotting (two subplots: left for error, right for time)
    size = 10
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(2*size, size))
    
    # Define colors for different alpha values
    colors = cm.viridis(np.linspace(0, 1, len(alphas)))
    
    # Iterate over different alpha values
    for idx, alpha in enumerate(alphas):
        # Calculate the analytical solution at t_end
        y_true = analytical_solution(t_end, alpha)
        print(f"Analytical solution y({t_end}) with alpha={alpha} = {y_true:.10f}\n")
        
        # Lists to store errors and computation times for each method and step size
        euler_errors = []
        heun_errors = []
        midpoint_errors = []
        euler_times = []
        heun_times = []
        midpoint_times = []
        
        # Print header
        print(f"Comparison of Methods for alpha = {alpha}:")
        print(f"{'Step Size h':>15} | {'Euler Error':>15} | {'Heun Error':>15} | {'Midpoint Error':>15}")
        print("-" * 75)
        
        # Apply each method for every step size
        for h in step_sizes:
            # Euler Method
            start_time = time.perf_counter()
            _, y_values_euler = euler_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            euler_time = end_time - start_time
            euler_times.append(euler_time)
            y_euler = y_values_euler[-1]
            euler_error = abs(y_euler - y_true) / abs(y_true)
            euler_errors.append(euler_error)

            # Heun Method
            start_time = time.perf_counter()
            _, y_values_heun = rk2_heun_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            heun_time = end_time - start_time
            heun_times.append(heun_time)
            y_heun = y_values_heun[-1]
            heun_error = abs(y_heun - y_true) / abs(y_true)
            heun_errors.append(heun_error)

            # Midpoint Method
            start_time = time.perf_counter()
            _, y_values_midpoint = rk2_midpoint_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            midpoint_time = end_time - start_time
            midpoint_times.append(midpoint_time)
            y_midpoint = y_values_midpoint[-1]
            midpoint_error = abs(y_midpoint - y_true) / abs(y_true)
            midpoint_errors.append(midpoint_error)

            # Print results for each step size
            print(f"{h:15.5e} | {euler_error:15.10e} | {heun_error:15.10e} | {midpoint_error:15.10e}")
        
        print("\n")  # Add space between different alpha values
        
        # Plot relative errors for each method on a log-log scale (left plot)
        ax1.loglog(step_sizes, euler_errors, 'o-', markersize=10, lw=4, label=f'Euler alpha={alpha}')
        ax1.loglog(step_sizes, heun_errors, 'x--', markersize=10, lw=4, label=f'Heun alpha={alpha}')
        ax1.loglog(step_sizes, midpoint_errors, 's-', markersize=10, lw=4, label=f'Midpoint alpha={alpha}')
        
        # Plot computation times for each method on a log-log scale (right plot)
        ax2.loglog(step_sizes, euler_times, 'o-', markersize=10, lw=4, label=f'Euler alpha={alpha}')
        ax2.loglog(step_sizes, heun_times, 'x--', markersize=10, lw=4,  label=f'Heun alpha={alpha}')
        ax2.loglog(step_sizes, midpoint_times, 's-', markersize=10, lw=4, label=f'Midpoint alpha={alpha}')
    
    # Set labels and titles for the left plot (relative error)
    ax1.set_xlabel('Step Size h')
    ax1.set_ylabel('Relative Error')
    ax1.set_title('Relative Error v.s. Step size')
    ax1.grid(True, which="both", ls="--")
    
    # Set labels and titles for the right plot (computation time)
    ax2.set_xlabel('Step Size h')
    ax2.set_ylabel('Computation Time (s)')
    ax2.set_title('Computation Time v.s. Step size')
    ax2.grid(True, which="both", ls="--")
    
    # Add legends
    ax1.legend()
    ax2.legend()
    
    # Display the plots
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()